# 台車モデルへ$\log$ バリア関数による制約を適用したC/GMRES

## 台車モデル

制御対象は以下の$x_1$方向にのみ速度$v$を持ち、$y_1$方向には速度を持たない台車とする。

<img src="images/nonholonomic_car.png" style="width:40%;"/>

台車の前進方向の運動と、回転方向の運動を以下とする。

$$
\begin{aligned}
m \dot{v}(t) = u(t) \\
I \dot{\omega}(t) = \tau(t) \\
\dot{\theta}(t) = \omega(t)
\end{aligned}
$$

$v$を $\Sigma_o$ で表現すると、次のようになる。

$$
\begin{aligned}
\dot{x}(t)= v(t) \cos(\theta(t)) \\
\dot{y}(t) = v(t) \sin(\theta(t))
\end{aligned}
$$

状態 $X$ を次のように設定する。

$$
X = \begin{bmatrix} v(t) & \theta(t) & \omega(t) & x(t) & y(t) \end{bmatrix}^T
$$

よって、状態方程式 $\dot{X} = f(X, U, t)$ は次の式となる。

$$
\dot{X} = \begin{bmatrix}
\dot{v}(t) \\ \dot{\theta}(t) \\ \dot{\omega}(t) \\ \dot{x}(t) \\ \dot{y}(t)
\end{bmatrix} = 
\begin{bmatrix}
u(t)/m \\\omega(t) \\ \tau(t)/I \\ v(t) \cos(\theta(t)) \\ v(t) \sin(\theta(t))
\end{bmatrix}
$$

$$
U=\begin{bmatrix}
u(t) \\ \tau(t)
\end{bmatrix}
$$

#### 非ホロノミック拘束について

台車は $\Sigma_1$ の $y_1$ 方向には横滑りしないという条件がある。

$\Sigma_o$ における台車の速度 $^o v$ を以下のようにベクトルで$x,y$軸それぞれの速度として表現する。

$$
^o v = \begin{bmatrix} \dot{x} \\ \dot{y} \end{bmatrix}
$$

$\Sigma_1$ の$y_1$ 方向の単位ベクトルを$e_{y_1}$とすると、これを$\Sigma_o$ で表すと以下となる。

$$
^o e_{y_1} = \begin{bmatrix} -\sin(\theta) \\ \cos(\theta) \end{bmatrix}
$$

よって、台車の$y_1$方向の速度は内積によって、以下のように表すことが出来る。

$$
v_{y_1} = {^o e_{y_1}}^T \ ^o v = -\dot{x} \sin(\theta) + \dot{y} \cos(\theta)
$$

台車は横滑りしないため、$v_{y_1}=0$であり、これより以下の拘束条件が導かれる。

$$
-\dot{x} \sin(\theta) + \dot{y} \cos(\theta) = 0
$$

これを非ホロノミック拘束と呼ぶ。

拘束条件を以下のように位置、姿勢だけで記述できる場合、これをホロノミック拘束と呼ぶ。

$$
g(x, y, \theta) = 0
$$

今回の拘束条件は以下のように速度に対するものである。

$$
-\dot{x} \sin(\theta) + \dot{y} \cos(\theta) = 0
$$

これを一般的には積分してホロノミック拘束$g(x,y,\theta) = 0$ という位置だけの拘束にはできない。

これは、「横方向に速度を出すことはできないが、横にある位置へ移動することできる」ということを意味している。

例えば台車の位置が$\Sigma_o$ で $(0,0)$ にあったとする。その後台車は左へ 90deg 旋回、1 前進、右へ-90deg 旋回とすれば $(0,1)$ へ到達することが出来る。

よって、位置が拘束されているのではなく、瞬間的に許される速度方向が拘束されれているということになる。

車体の前進方向の速度 $v$を$\Sigma_o$で表した方程式を上記の$v_{y_1}$ の式に代入すると以下のようになるため、状態方程式そのものが非ホロノミック拘束を常に満たしている。

$$
v_y = {^o e_{y_1}}^T \ ^o v = -\dot{x} \sin(\theta) + \dot{y} \cos(\theta) = -v \cos(\theta) \sin(\theta) + v \sin(\theta)\cos(\theta) = 0
$$

## 位置と制御入力の拘束

### 位置拘束

台車はXY平面を動く。そのため、ある範囲の中に入らないように位置の拘束を行う。

この範囲を以下の円の方程式から考える。

$$
(x - x_{c_1})^2 + (y - y_{c_1})^2 = {r_1}^2
$$

この方程式は、$(x_{c_1}, y_{c_1})$を中心に半径 $r_1$ の円である。

以下のように等式の拘束とすると、台車の位置 ($x,y$) は円周上に固定されることになる。

$$
(x - x_{c_1})^2 + (y - y_{c_1})^2 - {r_1}^2 = 0
$$

以下の方程式は、$(x_{c_1}, y_{c_1})$を中心に半径 $r_1$ の円の内側に台車の位置($x,y$)があることにある。これは範囲の外の出ないような拘束に繋がる。

$$
(x - x_{c_1})^2 + (y - y_{c_1})^2 \le {r_1}^2
$$

以下の方程式は、$(x_{c_1}, y_{c_1})$を中心に半径 $r_1$ の円の外側に台車の位置($x,y$)があることにある。これは範囲の中に入らないような高速に繋がる。

$$
(x - x_{c_1})^2 + (y - y_{c_1})^2 \ge {r_1}^2
$$

ここから G_1 の位置拘束をとする。

$$
G_1 = (x - x_{c_1})^2 + (y - y_{c_1})^2 - {r_1}^2 \ge 0
$$

例えば簡単に($x_{c_1},  y_{c_1}$) = ($0,0$)、$r_1 = 1$ として考えると、以下のように円周に正方向から近づくと、$G_1$ は境界の $0$ に近づいていく。

- $(x,y) = (1.1, 0)$ : $G_1 = 1.21 - 1 = 0.21 \ge 0$
- $(x,y) = (1.001, 0)$ : $G_1 = 1.001^2 - 1 = 0.002001 \ge 0$

よって、$-\log(G_1)$とすると、境界に近づくにつれ $+\infty$ と壁を高くすることが出来る。これをランニングコストに追加する。

この考え方で、$G_i \ (i = 1, \cdots, m)$ のように複数の位置拘束を考えることが出来る。

今回は位置拘束は2つとする。

$$
G_1 = (x - x_{c_1})^2 + (y - y_{c_1})^2 - {r_1}^2 \ge 0 \\
G_2 = (x - x_{c_2})^2 + (y - y_{c_2})^2 - {r_2}^2 \ge 0
$$

### 制御入力拘束

入力として $u, \tau$ の二つがある。そこで次のように上限値を設ける。

$$
\begin{split}
\begin{aligned}
u^2 &\le u^2_{max} \\
\tau^2 & \le \tau^2_{max}
\end{aligned}
\end{split}
$$

$\log$関数の特徴に合わせるように、次のように 正側の拘束に変換する。

$$
\begin{split}
\begin{aligned}
u^2 - u^2_{max}&\le 0  \\
\tau^2 - \tau^2_{max} & \le 0 
\end{aligned}
\end{split} \Rightarrow
\begin{split}
\begin{aligned}
u^2_{max} - u^2 &\ge 0  \\
\tau^2_{max} - \tau^2 & \ge 0 
\end{aligned}
\end{split}
$$

それぞれを以下のようにすると、

$$
G_u = u^2_{max} - u^2\\
G_\tau = \tau^2_{max} - \tau^2
$$

$\log$関数による制御入力拘束は以下となる。

$$
-\log(G_u) , \space -\log(G_\tau)
$$

位置と制御入力の拘束条件をまとめて、以下のように記述する。

$$
G(X,U) = \begin{bmatrix}
G_1 , G_2 , G_u, G_\tau
\end{bmatrix}^T
$$


## 拡大評価関数と各コスト関数



$\log$関数を制約条件に追加した拡大評価関数 $\bar{J}$ は次となる。

$$
\bar{J} = \Phi(X(T)) + \int_0^T \left[ L(X,U,t) - \eta^T \log(G(X,U)) + \lambda^T f(X,U,t)  \right] dt
$$

ここで$\lambda$は状態$X$に合わせて以下のようにする。

$$
\lambda = \begin{bmatrix}
\lambda_v , \lambda_\theta , \lambda_\omega ,\lambda_x, \lambda_y
\end{bmatrix}^T
$$

また、$\eta$は制約に合わせて以下のようにする。今回位置の制約は二つとする。

$$
\eta = \begin{bmatrix}
\eta_1 , \eta_2, \eta_u, \eta_\tau
\end{bmatrix}^T
$$

また、表現として$\log(G(X,U))$は以下のようになっているとする。

$$
\log(G(X,U)) = \begin{bmatrix}
\log(G_1) \\
\log(G_2) \\
\log(G_u) \\
\log(G_\tau)
\end{bmatrix}
$$

### 終端コスト $\Phi$

$$
\begin{aligned}
\Phi(X(T)) = \frac{1}{2} \Big[ q_{vT} \ (v(T))^2 + q_{\theta_T} \ (\theta(T) - \theta_{ref})^2 + q_{\omega_T}(\omega(T))^2 \\ 
+ q_{x_T}(x(T) - x_{ref})^2 + q_{y_T}(y(T) - y_{ref})^2 \Big]
\end{aligned}
$$ 

### ランニングコスト $L$